In [1]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Definiere den Modellnamen
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

# Lade das Modell und den Tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Definiere die Eingabe
eingabe = "Ich möchte ein Produkt kaufen."

# Tokenisiere die Eingabe
input_ids = tokenizer.encode(eingabe, return_tensors="pt")

# Rufe das Modell auf
antwort = model(input_ids)

# Drucke die Antwort
print(antwort)

c:\Users\alexs\Desktop\Bachelorprojekt\llm-tutor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1222.54it/s]


SequenceClassifierOutput(loss=None, logits=tensor([[ 2.3123, -2.0648]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ["KICONNECT_API_KEY"]
BASE_URL = os.environ["KICONNECT_BASE_URL"]
MODEL = os.environ["KICONNECT_MODEL"]

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

In [3]:
payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "user",
            "content": "Antworte nur mit: Verbindung erfolgreich."
        }
    ],
    "temperature": float(os.getenv("LLM_TEMPERATURE", "0")),
}
payload

{'model': 'HIER_MODELL_ID_EINTRAGEN',
 'messages': [{'role': 'user',
   'content': 'Antworte nur mit: Verbindung erfolgreich.'}],
 'temperature': 0.0}

In [4]:
import requests
response = requests.get(
    f"{BASE_URL}/models",
    headers=headers,
    timeout=30,
)

if response.status_code == 200:
    models = response.json().get("data", [])

    print("✅ API-Key funktioniert.")
    print("\nVerfügbare Modelle:")

    for model in models:
        print("-", model["id"])
else:
    print("❌ Anfrage fehlgeschlagen")
    print("Status:", response.status_code)
    print("Antwort:", response.text)

✅ API-Key funktioniert.

Verfügbare Modelle:
- Meta_Llama_3.1_8B_Instruct
- Qwen_3_30B_A3B_Instruct_2507
- Qwen_3_Omni_30B_A3B_Instruct
- Qwen_3.5_397B_A17B
- Qwen_3.8_27B
- E5_Mistral_7B_Instruct
- Multilingual_E5_Large_Instruct
- Qwen3-Embedding-4B


In [5]:
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

print("Konfiguration geladen.")

Konfiguration geladen.


In [6]:
response = requests.get(
    f"{BASE_URL}/models",
    headers=HEADERS,
    timeout=30,
)

print("HTTP-Status:", response.status_code)

if response.ok:
    models = response.json().get("data", [])

    print(f"✅ Zugriff funktioniert – {len(models)} Modelle gefunden:\n")

    for index, model in enumerate(models, start=1):
        print(f"{index}: {model['id']}")
else:
    print("❌ Zugriff fehlgeschlagen:")
    print(response.text)

HTTP-Status: 200
✅ Zugriff funktioniert – 8 Modelle gefunden:

1: Meta_Llama_3.1_8B_Instruct
2: Qwen_3_30B_A3B_Instruct_2507
3: Qwen_3_Omni_30B_A3B_Instruct
4: Qwen_3.5_397B_A17B
5: Qwen_3.8_27B
6: E5_Mistral_7B_Instruct
7: Multilingual_E5_Large_Instruct
8: Qwen3-Embedding-4B


In [7]:
models

[{'id': 'Meta_Llama_3.1_8B_Instruct',
  'object': 'model',
  'created': 1792324315,
  'owned_by': 'fh-offenburg.de'},
 {'id': 'Qwen_3_30B_A3B_Instruct_2507',
  'object': 'model',
  'created': 1792324315,
  'owned_by': 'fh-offenburg.de'},
 {'id': 'Qwen_3_Omni_30B_A3B_Instruct',
  'object': 'model',
  'created': 1792324315,
  'owned_by': 'fh-offenburg.de'},
 {'id': 'Qwen_3.5_397B_A17B',
  'object': 'model',
  'created': 1792324315,
  'owned_by': 'fh-offenburg.de'},
 {'id': 'Qwen_3.8_27B',
  'object': 'model',
  'created': 1792324315,
  'owned_by': 'fh-offenburg.de'},
 {'id': 'E5_Mistral_7B_Instruct',
  'object': 'model',
  'created': 1792324315,
  'owned_by': 'fh-offenburg.de'},
 {'id': 'Multilingual_E5_Large_Instruct',
  'object': 'model',
  'created': 1792324315,
  'owned_by': 'fh-offenburg.de'},
 {'id': 'Qwen3-Embedding-4B',
  'object': 'model',
  'created': 1792324315,
  'owned_by': 'fh-offenburg.de'}]

In [ ]:
#auswahl = int(input("Nummer des Modells: ")) - 1
#MODEL = models[auswahl]["id"]
MODEL = models[0]["id"]
print("Ausgewähltes Modell:", MODEL)

Ausgewähltes Modell: Meta_Llama_3.1_8B_Instruct


In [9]:
payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "system",
            "content": "Du antwortest kurz und präzise."
        },
        {
            "role": "user",
            "content": "Antworte exakt mit: KI:connect funktioniert."
        }
    ],
    "temperature": 0,
}

response = requests.post(
    f"{BASE_URL}/chat/completions",
    headers=HEADERS,
    json=payload,
    timeout=120,
)

print("HTTP-Status:", response.status_code)

if response.ok:
    result = response.json()

    print("\n✅ Modellantwort:")
    print(result["choices"][0]["message"]["content"])

    print("\nVerwendetes Modell:")
    print(result.get("model"))

    print("\nTokenverbrauch:")
    print(result.get("usage"))
else:
    print("\n❌ Modellaufruf fehlgeschlagen:")
    print(response.text)

HTTP-Status: 200

✅ Modellantwort:
KI:connect funktioniert.

Verwendetes Modell:
meta-llama-3.1-8b-instruct

Tokenverbrauch:
{'prompt_tokens': 60, 'total_tokens': 67, 'completion_tokens': 7, 'prompt_tokens_details': None}


In [11]:
import requests
import json

# API-Schlüssel und -Endpoint


# Konfiguration
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

# Ausgewähltes Modell
MODEL = "Meta_Llama_3.1_8B_Instruct"

# Payload für die Chat-Completions-Anfrage
payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "system",
            "content": "Du antwortest kurz und präzise."
        },
        {
            "role": "user",
            "content": "Antworte exakt mit: Die Antwort ist 42."
        }
    ],
    "temperature": 0,
}

# API-Anfrage
response = requests.post(
    f"{BASE_URL}/chat/completions",
    headers=HEADERS,
    json=payload,
    timeout=120,
)

# Antwort verarbeiten
if response.ok:
    result = response.json()

    # Modellantwort ausgeben
    print("\n✅ Modellantwort:")
    print(result["choices"][0]["message"]["content"])

    # Verwendetes Modell ausgeben
    print("\nVerwendetes Modell:")
    print(result.get("model"))

    # Tokenverbrauch ausgeben
    print("\nTokenverbrauch:")
    print(result.get("usage"))
else:
    print("\n❌ Modellaufruf fehlgeschlagen:")
    print(response.text)


✅ Modellantwort:
Die Antwort ist 42.

Verwendetes Modell:
meta-llama-3.1-8b-instruct

Tokenverbrauch:
{'prompt_tokens': 59, 'total_tokens': 66, 'completion_tokens': 7, 'prompt_tokens_details': None}
